# Train Origin Inference Baseline (LightGBM)

This notebook demonstrates a simple TF-IDF + LightGBM baseline to predict product origin country using labeled `OriginLabel` data from the Django app. It shows setup steps, data extraction using Django ORM, training, evaluation, and saving artifacts to `core/ml/models/`.

**Notes:** Install optional dependencies first: `pip install lightgbm scikit-learn joblib`

## 1) Setup Django environment (in-notebook)

import os
import django
# Adjust if your settings module is named differently
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'Nexus.settings')
django.setup()
print('Django configured:', os.environ.get('DJANGO_SETTINGS_MODULE'))

## 2) Import libraries & extract labeled data

from core.models import OriginLabel, Product
from collections import Counter

# Build dataset where label_country is set
rows = []
for ol in OriginLabel.objects.filter(label_country__isnull=False).select_related('product'):
    text = ' '.join(filter(None, [ol.product.name, ol.product.description or '', ol.product.keywords_for_ai or '', getattr(ol.product.vendor, 'location_country', '')]))
    rows.append((text, str(ol.label_country)))

print('Total labeled examples:', len(rows))
print('Per-class counts:', Counter([r[1] for r in rows]))

## 3) Vectorize (TF-IDF) and prepare train/test splits

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import classification_report
    import joblib
    import lightgbm as lgb
except Exception as e:
    raise RuntimeError('Please install optional ML deps: pip install lightgbm scikit-learn joblib')

texts = [t for t, _ in rows]
labels = [l for _, l in rows]

# Filter small classes if needed (example threshold: 5)
from collections import Counter
counts = Counter(labels)
valid = {c for c, cnt in counts.items() if cnt >= 5}
if valid:
    filtered = [(t, l) for (t, l) in zip(texts, labels) if l in valid]
    texts, labels = zip(*filtered)

vect = TfidfVectorizer(min_df=1, max_features=20000, ngram_range=(1, 2))
X = vect.fit_transform(texts)
le = LabelEncoder()
y = le.fit_transform(labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('Train size:', X_train.shape, 'Test size:', X_test.shape)

## 4) Train LightGBM

params = {'objective': 'multiclass', 'num_class': len(le.classes_), 'metric': 'multi_logloss', 'verbosity': -1, 'seed': 42}
dtrain = lgb.Dataset(X_train, label=y_train)
dtest = lgb.Dataset(X_test, label=y_test, reference=dtrain)
model = lgb.train(params, dtrain, num_boost_round=100, valid_sets=[dtrain, dtest], early_stopping_rounds=10, verbose_eval=10)

preds = model.predict(X_test)
y_pred = preds.argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=list(le.classes_)))

## 5) Save artifacts

import os
os.makedirs('core/ml/models', exist_ok=True)
model_path = 'core/ml/models/origin_lightgbm.txt'
vect_path = 'core/ml/models/origin_vectorizer.joblib'
enc_path = 'core/ml/models/origin_label_encoder.joblib'
meta_path = 'core/ml/models/train_metadata.json'
model.save_model(model_path)
joblib.dump(vect, vect_path)
joblib.dump(le, enc_path)
import json
meta = {'trained_at': __import__('datetime').datetime.utcnow().isoformat() + 'Z', 'classes': list(le.classes_), 'params': params}
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)
print('Artifacts saved to core/ml/models/')

## 6) Simple predict example

sample_text = 'Handmade wood toy. Made in Germany.'
Xs = vect.transform([sample_text])
proba = model.predict(Xs)[0]
idx = int(proba.argmax())
print('Predicted country:', le.inverse_transform([idx])[0], 'confidence=', proba[idx])

## Next steps & tips
- Add more features: product images (image model), GTIN/brand lookups, structured vendor metadata.
- Add cross-validation & hyperparameter tuning (Optuna or grid search).
- Track experiments with MLflow or a simple artifacts registry.